In [ ]:
import os
import glob
import pandas as pd
import marimo as mo
import xgboost as xgb
import kagglehub
import joblib  
from sentence_transformers import SentenceTransformer  
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error


os.environ["KAGGLE_API_TOKEN"] = "your_kaggle_token_here"
# Run to wherever you want to put the data(I put it in my external hardrive)
os.environ["KAGGLEHUB_CACHE"] = "/run/media/s5812886/T7 Shield/kagglehub_cache"

print("Libraries loaded")

In [ ]:
print("Downloading YouTube dataset")
print("(This may take a few minutes. It will unzip by itself)")

yt_dataset_path = kagglehub.dataset_download("datasnaek/youtube-new")

print(f" Download complete!")
print(f" Files extracted to: {yt_dataset_path}")

In [ ]:
print("Loading ONLY English YouTube Data...")

# We only load the English-speaking regions!
english_files = ["USvideos.csv", "GBvideos.csv", "CAvideos.csv"]
yt_csv_files = [f"{yt_dataset_path}/{f}" for f in english_files]

yt_useful_cols = [
    "title", "publish_time", "views", "likes", 
    "dislikes", "comment_count", "category_id"
]

yt_df_raw = pd.concat(
    [pd.read_csv(f, usecols=yt_useful_cols, encoding='latin-1') for f in yt_csv_files],
    ignore_index=True
)

print("Step 3: Cleaning & Sorting the Data...")
yt_df_clean = yt_df_raw.dropna(subset=["title", "views"]).copy()
yt_df_clean = yt_df_clean[yt_df_clean["views"] > 0].reset_index(drop=True)

yt_df_clean["publish_time"] = pd.to_datetime(yt_df_clean["publish_time"], format='mixed', utc=True)
yt_df_clean = yt_df_clean.sort_values(by="publish_time", ascending=False).reset_index(drop=True)

print(f" Final English Dataset: {len(yt_df_clean):,} rows")
yt_df_clean.head(5)

In [ ]:
print(" Calculating YouTube Engagement Score")
yt_df_clean['engagement_total'] = yt_df_clean['likes'] + yt_df_clean['comment_count']
yt_df_clean['virality_score'] = (yt_df_clean['engagement_total'] / yt_df_clean['views']) * 100
yt_df_clean['virality_score'] = yt_df_clean['virality_score'].clip(upper=20)

print(" Slicing the training sample")
df_train_yt = yt_df_clean.sample(n=min(100000, len(yt_df_clean)), random_state=42)

X_text_yt = df_train_yt['title'].tolist()
y_yt = df_train_yt['virality_score']

# this is the swap
print(" Step 3: Generating Sentence Embeddings ")
print(" This will take 10-20 mins on CPU.")

embedder = SentenceTransformer('all-MiniLM-L6-v2')
X_numeric_yt = embedder.encode(
    X_text_yt,
    batch_size=64,
    show_progress_bar=True
)

print(f" Embeddings shape: {X_numeric_yt.shape}")
# Each title is now 384 numbers representing its meaning too

print(" Step 4: Training XGBoost on Meaning-Based Features")
X_train_yt, X_test_yt, y_train_yt, y_test_yt = train_test_split(
    X_numeric_yt, y_yt, test_size=0.2, random_state=42
)

xgb_model_yt = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=12,
    min_child_weight=1,
    random_state=42,
    n_jobs=-1
)
xgb_model_yt.fit(X_train_yt, y_train_yt)

mae = mean_absolute_error(y_test_yt, xgb_model_yt.predict(X_test_yt))
print(f" Training Complete! MAE: {mae:.4f}")

# test
print("\n Testing the Meaning-Aware Scorer:")
yt_test_titles = [
    "I Built a Secret Hidden Room in my House",
    "Teaching my friend how to draw for the first time",
    "How to fix a leaking sink in 5 minutes"
]

yt_test_embeddings = embedder.encode(yt_test_titles)
yt_test_scores = xgb_model_yt.predict(yt_test_embeddings)

for title, score in zip(yt_test_titles, yt_test_scores):
    print(f"Score: {score:.2f} | Title: '{title}'")

# SAVE BOTH LOCALLY

print("\n Saving AI brain locally")

save_path = "/home/s5812886/Desktop/Youtube-Title-Generator-AI/AITEST"

joblib.dump(xgb_model_yt, f'{save_path}/youtube_model.pkl')
joblib.dump(embedder, f'{save_path}/youtube_embedder.pkl')

print(f"Saved to {save_path}")

In [ ]:
print("Saving the AI Brain to your hard drive")

# This saves the AI's math and dictionary into tiny files
model = joblib.load("/home/s5812886/Desktop/Youtube-Title-Generator-AI/AITEST/youtube_model.pkl")
embedders = joblib.load("/home/s5812886/Desktop/Youtube-Title-Generator-AI/AITEST/youtube_embedder.pkl")

print("Saved successfully")